In [1]:
import os
import time
import mne
import numpy as np
import pandas as pd

import bsl
from bsl import StreamPlayer, datasets
# from bsl.externals import pylsl  # distributed version of pylsl
from bsl.triggers import TriggerDef

import pylsl

import pickle

import math
import matplotlib
import matplotlib.pyplot as plt
from pythonosc.udp_client import SimpleUDPClient

In [3]:
from pylsl import resolve_streams

streams = resolve_streams()
for s in streams:
    print(repr(s.name()))  # repr() shows exact string with no hidden characters

'OxySoft'
'OxySoft Event Marker'


In [10]:
from pylsl import resolve_streams, StreamInlet

streams = resolve_streams()
fnirs_stream = [s for s in streams if s.type() == 'NIRS'][0]
inlet = StreamInlet(fnirs_stream)
info = inlet.info()

print("sfreq:", info.nominal_srate())
print("channel_count:", info.channel_count())

ch = info.desc().child('channels').child('channel')
while ch.name() == 'channel':
    print(ch.child_value('label'), ch.child_value('type'), ch.child_value('unit'))
    ch = ch.next_sibling()

sfreq: 75.0
channel_count: 28
Rx1 - Tx1 O2Hb NIRS umol
Rx1 - Tx1 HHb NIRS umol
Rx1 - Tx3 O2Hb NIRS umol
Rx1 - Tx3 HHb NIRS umol
Rx2 - Tx1 O2Hb NIRS umol
Rx2 - Tx1 HHb NIRS umol
Rx2 - Tx3 O2Hb NIRS umol
Rx2 - Tx3 HHb NIRS umol
Rx3 - Tx4 O2Hb NIRS umol
Rx3 - Tx4 HHb NIRS umol
Rx3 - Tx5 O2Hb NIRS umol
Rx3 - Tx5 HHb NIRS umol
Rx8 - Tx9 O2Hb NIRS umol
Rx8 - Tx9 HHb NIRS umol
Rx8 - Tx10 O2Hb NIRS umol
Rx8 - Tx10 HHb NIRS umol
Rx5 - Tx6 O2Hb NIRS umol
Rx5 - Tx6 HHb NIRS umol
Rx5 - Tx8 O2Hb NIRS umol
Rx5 - Tx8 HHb NIRS umol
Rx6 - Tx6 O2Hb NIRS umol
Rx6 - Tx6 HHb NIRS umol
Rx6 - Tx8 O2Hb NIRS umol
Rx6 - Tx8 HHb NIRS umol
Rx4 - Tx2 O2Hb NIRS umol
Rx4 - Tx2 HHb NIRS umol
Rx7 - Tx7 O2Hb NIRS umol
Rx7 - Tx7 HHb NIRS umol


In [14]:
streams = resolve_streams(wait_time=3.0)
print(streams)
for s in streams:
    print(s.name(), s.type())

[<StreamInfo name='OxySoft', type='NIRS', channels=28, srate=75.0, format=1, source_id='OxySoft'>, <StreamInfo name='OxySoft Event Marker', type='Markers', channels=1, srate=0.0, format=3, source_id='OxySoft'>]
OxySoft NIRS
OxySoft Event Marker Markers


In [ ]:
receiver = bsl.StreamReceiver(bufsize=10, winsize=10, stream_name=['OxySoft'])


In [15]:
print(receiver.streams)

{}


In [17]:
from pylsl import resolve_streams, StreamInlet
import numpy as np

streams = resolve_streams()
fnirs_stream = [s for s in streams if s.name() == 'OxySoft'][0]
inlet = StreamInlet(fnirs_stream)

# Pull a chunk
samples, timestamps = inlet.pull_chunk(timeout=1.0)
data = np.array(samples)
print(data.shape)  # should be (n_samples, 28)

(75, 28)


In [11]:
ch_names = [
    'Rx1 - Tx1 O2Hb', 'Rx1 - Tx1 HHb',
    'Rx1 - Tx3 O2Hb', 'Rx1 - Tx3 HHb',
    'Rx2 - Tx1 O2Hb', 'Rx2 - Tx1 HHb',
    'Rx2 - Tx3 O2Hb', 'Rx2 - Tx3 HHb',
    'Rx3 - Tx4 O2Hb', 'Rx3 - Tx4 HHb',
    'Rx3 - Tx5 O2Hb', 'Rx3 - Tx5 HHb',
    'Rx8 - Tx9 O2Hb', 'Rx8 - Tx9 HHb',
    'Rx8 - Tx10 O2Hb', 'Rx8 - Tx10 HHb',
    'Rx5 - Tx6 O2Hb', 'Rx5 - Tx6 HHb',
    'Rx5 - Tx8 O2Hb', 'Rx5 - Tx8 HHb',
    'Rx6 - Tx6 O2Hb', 'Rx6 - Tx6 HHb',
    'Rx6 - Tx8 O2Hb', 'Rx6 - Tx8 HHb',
    'Rx4 - Tx2 O2Hb', 'Rx4 - Tx2 HHb',
    'Rx7 - Tx7 O2Hb', 'Rx7 - Tx7 HHb',
]

fnirs_info = mne.create_info(
    ch_names=ch_names,
    sfreq=75.0,
    ch_types=['fnirs_cw_amplitude'] * 28
)

for ch in fnirs_info['chs']:
    ch['cal'] = 1e-6

In [13]:
print(receiver.streams)

{}


In [ ]:
from pylsl import resolve_streams, StreamInlet
from collections import deque
import numpy as np
import time

# ── Constants ────────────────────────────────────────────────────────────────
FS = 75.0
WINDOW_S = 10
WINDOW_SAMPLES = int(FS * WINDOW_S)  # 750 samples
UPDATE_EVERY_S = 0.5                  # run classification every 500ms
N_CHANNELS = 28

HBO_COLS = list(range(0, N_CHANNELS, 2))  # 0,2,4,...
HBR_COLS = list(range(1, N_CHANNELS, 2))  # 1,3,5,...

# ── LSL inlet ────────────────────────────────────────────────────────────────
streams = resolve_streams()
fnirs_stream = [s for s in streams if s.name() == 'OxySoft'][0]
inlet = StreamInlet(fnirs_stream)

# ── Ring buffer ──────────────────────────────────────────────────────────────
buffer = deque(maxlen=WINDOW_SAMPLES)

# ── Main loop ────────────────────────────────────────────────────────────────
last_process_time = time.time()

while True:
    samples, _ = inlet.pull_chunk(timeout=0.1)
    if samples:
        buffer.extend(samples)

    if len(buffer) < WINDOW_SAMPLES:
        print(f"Buffering... {len(buffer)}/{WINDOW_SAMPLES} samples")
        continue

    now = time.time()
    if now - last_process_time < UPDATE_EVERY_S:
        continue
    last_process_time = now

    # ── Feature extraction ───────────────────────────────────────────────────
    data = np.array(buffer)                      # (750, 28)
    data = np.nan_to_num(data)

    hbo = data[:, HBO_COLS]                      # (750, 14)
    hbr = data[:, HBR_COLS]                      # (750, 14)

    hbo_mean = hbo.mean(axis=0)                  # (14,)
    hbr_mean = hbr.mean(axis=0)                  # (14,)

    t = np.arange(WINDOW_SAMPLES) / FS
    hbo_slope = np.polyfit(t, hbo, 1)[0]        # (14,)
    hbr_slope = np.polyfit(t, hbr, 1)[0]        # (14,)

    features = np.concatenate([hbo_mean, hbr_mean, hbo_slope, hbr_slope])  # (56,)
    print("Features shape:", features.shape)
    print("HbO mean (first 3 channels):", hbo_mean[:3])
    print("HbR mean (first 3 channels):", hbr_mean[:3])

In [19]:
features

[]